# Objective 1 — Supplementary P0 Incremental Control Execution Engine

**Authoritative Execution Path for Policy P0 (Incremental-Only Baseline Control)**  

### Methodological Foundation
- **P0 Definition**: Continuous chronological stream, predict-then-train prequential evaluation, unconditional incremental `learn_one()` updates upon ground-truth label revelation.
- **Confound Control**: NO periodic retraining, NO drift-triggered retraining, NO model replacements, NO retraining window parameter $W$.
- **Adaptation Count**: Strictly zero ($A = 0$).
- **Fairness**: Shares exact data ingestion, chronological verification, warmup split ($15\% = 88,581$), feature encoding, and initial learner initialization as policies P1–P3.

## 1. Import Required Libraries and Environment Setup

Load canonical thesis libraries, configure deterministic random seeds, and establish paths.

In [1]:
import os
import sys
import time
import json
import hashlib
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

Python Version: 3.12.4
Pandas Version: 3.0.3
NumPy Version:  2.5.1


## 2. Execution Mode Selection & Job Configuration

Choose between `SMOKE` (fast local verification subset) and `FULL` (authoritative 590,540 transaction stream).

In [2]:
# TOP-LEVEL EXECUTION MODE SELECTION
# Options:
#   "SMOKE" : Fast local verification mode (1,000 rows)
#   "FULL"  : Authoritative Kaggle/Production execution (590,540 rows)
RUN_MODE = "SMOKE"

# SHARED HYPERPARAMETERS (Locked by DEC-01, DEC-02, DEC-04)
GRACE_PERIOD = 50 if RUN_MODE == "SMOKE" else 200
DELTA = 0.002
WARMUP_FRAC = 0.15
DIAGNOSTIC_HORIZON = 500

if RUN_MODE == "SMOKE":
    SMOKE_ROW_LIMIT = 1000
    ROLLING_WINDOW_SIZE = 200
    MAX_TRAJECTORY_POINTS = 50
    SEEDS = [42]
else:
    SMOKE_ROW_LIMIT = None
    ROLLING_WINDOW_SIZE = 5000
    MAX_TRAJECTORY_POINTS = 200
    SEEDS = [42, 101]  # 42 (primary), 101 (reproducibility check)

# P0 Job Specifications: Window size is None (not applicable to incremental control)
JOB_SPECS = [
    {"policy": "P0", "window_size": None, "no_swap": False, "seed": s}
    for s in SEEDS
]

print(f"Configured RUN_MODE: {RUN_MODE}")
print(f"Total P0 Target Jobs: {len(JOB_SPECS)}")
for spec in JOB_SPECS:
    print(f"  - Policy {spec['policy']} | window_size={spec['window_size']} | seed={spec['seed']}")

Configured RUN_MODE: SMOKE
Total P0 Target Jobs: 1
  - Policy P0 | window_size=None | seed=42


## 3. Project Root & Directory Hierarchy Configuration

Resolve repository root and configure the canonical outputs hierarchy (`outputs/checkpoints/objective1_runs/`).

In [3]:
def find_project_root(start_path: Path) -> Optional[Path]:
    current = start_path.resolve()
    while current != current.parent:
        if (current / 'src').is_dir() and (current / 'pyproject.toml').is_file():
            return current
        current = current.parent
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
kaggle_working = Path('/kaggle/working')
if kaggle_working.exists() and (PROJECT_ROOT is None or not (PROJECT_ROOT / 'src').is_dir()):
    REPO_NAME = 'adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability'
    REPO_DIR = kaggle_working / REPO_NAME
    PROJECT_ROOT = REPO_DIR

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root resolved: {PROJECT_ROOT}')

# Outputs directory hierarchy matching canonical Objective 1 structure
OUTPUTS_BASE = kaggle_working if kaggle_working.exists() else PROJECT_ROOT
CHECKPOINT_DIR = OUTPUTS_BASE / 'outputs' / 'checkpoints' / 'objective1_runs'
MANIFEST_PATH = OUTPUTS_BASE / 'outputs' / 'checkpoints' / 'objective1_manifest.json'
ZIP_ARCHIVE_PATH = OUTPUTS_BASE / 'outputs' / 'objective1_artifacts_live.zip'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Manifest path:        {MANIFEST_PATH}')
print(f'Live archive mirror:  {ZIP_ARCHIVE_PATH}')

Project root resolved: C:\Projects\Thesis
Checkpoint directory: C:\Projects\Thesis\outputs\checkpoints\objective1_runs
Manifest path:        C:\Projects\Thesis\outputs\checkpoints\objective1_manifest.json
Live archive mirror:  C:\Projects\Thesis\outputs\objective1_artifacts_live.zip


## 4. Benchmark Dataset Discovery & Ingestion

Locate and ingest IEEE-CIS transaction and identity data using canonical loader.

In [4]:
def resolve_ieee_cis_dir() -> Path:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob('train_transaction.csv'))
        if matches:
            return matches[0].parent
    for base in [PROJECT_ROOT, Path.cwd(), Path('/kaggle/working')]:
        if base.exists():
            matches = list(base.rglob('train_transaction.csv'))
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Could not locate 'train_transaction.csv'.")

IEEE_CIS_DIR = resolve_ieee_cis_dir()
print(f'IEEE-CIS data source: {IEEE_CIS_DIR}')

from src.data.loader import load_ieee_cis
from src.utils.seed import set_seed

df_raw = load_ieee_cis(data_dir=IEEE_CIS_DIR, split='train', join_identity=True)
if SMOKE_ROW_LIMIT is not None:
    df_raw = df_raw.iloc[:SMOKE_ROW_LIMIT].copy()
print(f'Ingested raw transactions: {len(df_raw):,} rows, {len(df_raw.columns)} columns')

IEEE-CIS data source: C:\Projects\Thesis\datasets\ieee_cis
Ingested raw transactions: 1,000 rows, 434 columns


## 5. Chronological Verification & Warmup-Only Preprocessing

Verify strict temporal monotonicity and fit preprocessor strictly on warmup data.

In [5]:
dt_series = df_raw['TransactionDT']
is_monotonic = bool(dt_series.is_monotonic_increasing)
print(f"Chronological Monotonicity Check: {is_monotonic}")
assert is_monotonic, "Dataset MUST be sorted monotonically by TransactionDT!"

data_fingerprint = hashlib.sha256(
    f"{len(df_raw)}_{df_raw['TransactionID'].iloc[0]}_{df_raw['TransactionID'].iloc[-1]}".encode('utf-8')
).hexdigest()[:16]
print(f"Dataset fingerprint: {data_fingerprint}")

n_total = len(df_raw)
warmup_size = int(n_total * WARMUP_FRAC)
stream_size = n_total - warmup_size
print(f"Warmup size (Phase 1): {warmup_size:,} ({WARMUP_FRAC*100:.1f}%)")
print(f"Stream size (Phase 2): {stream_size:,} ({(1-WARMUP_FRAC)*100:.1f}%)")

from src.data.preprocessing import StreamingPreprocessor

preprocessor = StreamingPreprocessor(dataset_name="ieee_cis", scale_features=False)
print("Fitting streaming preprocessor on warmup partition only...")
preprocessor.fit(df_raw.iloc[:warmup_size])
print(f"Preprocessor fitted. Total features: {len(preprocessor.feature_names):,}")

print("Transforming full dataset...")
X_all, y_all, seg_all = preprocessor.transform(df_raw)
print(f"Feature matrix X: {X_all.shape}")
print(f"Target Series y:  {len(y_all):,} (fraud rate = {y_all.mean():.4f})")

Chronological Monotonicity Check: True
Dataset fingerprint: af2d52cfe83db380
Warmup size (Phase 1): 150 (15.0%)
Stream size (Phase 2): 850 (85.0%)
Fitting streaming preprocessor on warmup partition only...
Preprocessor fitted. Total features: 431
Transforming full dataset...
Feature matrix X: (1000, 431)
Target Series y:  1,000 (fraud rate = 0.0150)


## 6. Manifest Initialization & Resumption Discovery

Initialize ExperimentManifest to manage job state and atomic persistence.

In [6]:
from src.pipeline.manifest import ExperimentManifest

manifest = ExperimentManifest(
    manifest_path=MANIFEST_PATH,
    artifacts_dir=CHECKPOINT_DIR,
    zip_archive_path=ZIP_ARCHIVE_PATH,
    experiment_name="Objective1_E1_E2",
    dataset_name="IEEE-CIS",
)
print(f"Loaded manifest with {len(manifest.jobs)} known jobs.")

Loaded manifest with 3 known jobs.


## 7. Trajectory Downsampling Helper

Extract rolling PR-AUC trajectory downsampled to compact representation for serialization.

In [7]:
from sklearn.metrics import precision_recall_curve, auc

def extract_downsampled_trajectories(records: List[Any], window_size: int = 5000, max_points: int = 150) -> Dict[str, Any]:
    n_records = len(records)
    if n_records == 0:
        return {"indices": [], "rolling_pr_auc": []}
    step = max(1, n_records // max_points)
    indices = []
    pr_aucs = []
    y_true_all = [r.y_true for r in records]
    y_prob_all = [r.y_prob for r in records]
    for end_idx in range(window_size, n_records + 1, step):
        start_idx = end_idx - window_size
        y_w = y_true_all[start_idx:end_idx]
        p_w = y_prob_all[start_idx:end_idx]
        if sum(y_w) > 0 and len(y_w) - sum(y_w) > 0:
            prec, rec, _ = precision_recall_curve(y_w, p_w)
            score = float(auc(rec, prec))
        else:
            score = 0.0
        indices.append(end_idx)
        pr_aucs.append(round(score, 5))
    return {
        "indices": indices,
        "rolling_pr_auc": pr_aucs,
        "window_size": window_size,
    }
print("Trajectory downsampling helper defined.")

Trajectory downsampling helper defined.


## 8. Authoritative Job Execution Loop (Resumable & Atomic)

Execute P0 incremental control loop across configured seeds.

In [8]:
from src.pipeline.runner import PrequentialRunner

total_jobs = len(JOB_SPECS)
job_idx = 0

print(f"Starting execution loop over {total_jobs} P0 jobs...")

for spec in JOB_SPECS:
    job_idx += 1
    policy = spec["policy"]
    seed = spec["seed"]
    window_size = spec["window_size"]
    no_swap = spec.get("no_swap", False)
    job_id = manifest.make_job_id(policy, seed, window_size=window_size, no_swap=no_swap)

    if manifest.is_job_completed(policy, seed, window_size=window_size, no_swap=no_swap):
        print(f"[{job_idx}/{total_jobs}] Job {job_id} already COMPLETED on disk -> SKIPPING.")
        continue

    print(f"[{job_idx}/{total_jobs}] Executing Job {job_id}...")
    manifest.record_start(policy, seed, dataset_fingerprint=data_fingerprint, window_size=window_size, no_swap=no_swap)
    set_seed(seed)

    config_snapshot = {
        "warmup_size": warmup_size,
        "stream_size": stream_size,
        "warmup_frac": WARMUP_FRAC,
        "window_size": window_size,
        "n_interval": 10000,
        "grace_period": GRACE_PERIOD,
        "delta": DELTA,
        "diagnostic_horizon": DIAGNOSTIC_HORIZON,
        "no_swap": no_swap,
        "run_mode": RUN_MODE,
    }

    t_start = time.perf_counter()
    try:
        runner = PrequentialRunner.from_config(
            policy_str=policy,
            detector_str="adwin",
            grace_period=GRACE_PERIOD,
            delta=DELTA,
            window_size=window_size,
            detector_kwargs={"delta": DELTA},
            segment_aware=False,
            seed=seed,
            p0_mode="incremental",
            diagnostic_horizon=DIAGNOSTIC_HORIZON,
            no_swap=no_swap,
        )

        run_result = runner.run(
            X=X_all,
            y=y_all,
            segment=seg_all,
            warmup_size=warmup_size,
        )
        wall_clock_s = time.perf_counter() - t_start

        trajectories = extract_downsampled_trajectories(
            run_result.records,
            window_size=ROLLING_WINDOW_SIZE,
            max_points=MAX_TRAJECTORY_POINTS,
        )

        artifact_path = manifest.record_completion(
            policy=policy,
            seed=seed,
            run_result=run_result,
            wall_clock_duration_s=wall_clock_s,
            dataset_fingerprint=data_fingerprint,
            code_version="authoritative_o1_v3_p0_supplement",
            config_snapshot=config_snapshot,
            trajectories=trajectories,
            window_size=window_size,
            no_swap=no_swap,
        )

        fm = run_result.final_metrics
        lp = run_result.latency_percentiles
        pr_val = fm.pr_auc if hasattr(fm, "pr_auc") else fm.get("pr_auc", 0.0)
        roc_val = fm.roc_auc if hasattr(fm, "roc_auc") else fm.get("roc_auc", 0.0)
        p95_val = lp.get("p95_ms", lp.get("p95", 0.0) * 1000)
        print(
            f"    [COMPLETED] Duration: {wall_clock_s:.1f}s | "
            f"PR-AUC: {pr_val:.5f} | ROC-AUC: {roc_val:.5f} | "
            f"Adaptations: {len(run_result.adaptation_log)} | "
            f"Latency p95: {p95_val:.2f}ms | Saved: {artifact_path.name}"
        )
    except Exception as ex:
        manifest.record_failure(policy, seed, str(ex), window_size=window_size, no_swap=no_swap)
        print(f"    [FAILED] Job {job_id} encountered error: {ex}")
        raise

print("P0 execution loop complete.")

Starting execution loop over 1 P0 jobs...
[1/1] Executing Job IEEE-CIS_Objective1_E1_E2_P0_seed42...
    [COMPLETED] Duration: 6.6s | PR-AUC: 0.01274 | ROC-AUC: 0.32371 | Adaptations: 0 | Latency p95: 0.07ms | Saved: run_P0_seed42.json
P0 execution loop complete.


## 9. Execution Summary Table & Historical Validation

Display completed jobs summary and reconcile against historical benchmark.

In [10]:
summary_df = manifest.get_summary_dataframe()
print("P0 Manifest Summary Table")
print(summary_df.to_string() if not summary_df.empty else "No completed jobs found.")

# Historical reference validation (PR-AUC ≈ 0.25637 on full 501,959 stream)
HISTORICAL_P0_PR_AUC = 0.25637245056608454
if RUN_MODE == "FULL":
    p0_job = manifest.jobs.get("IEEE-CIS_Objective1_E1_E2_P0_seed42")
    if p0_job and p0_job.pr_auc is not None:
        delta = abs(p0_job.pr_auc - HISTORICAL_P0_PR_AUC)
        print(f"\nHistorical Reconciliation Check (Seed 42):")
        print(f"  Executed PR-AUC:   {p0_job.pr_auc:.7f}")
        print(f"  Historical PR-AUC: {HISTORICAL_P0_PR_AUC:.7f}")
        print(f"  Absolute Delta:    {delta:.8f}")
        assert delta < 1e-4, f"Discrepancy detected vs historical reference: {delta}"
        print("  [RECONCILIATION VERIFIED] Executed P0 matches historical reference.")

P0 Manifest Summary Table
                                       job_id        experiment   dataset policy  seed     status                        start_time                          end_time  wall_clock_duration_s dataset_fingerprint                       code_version    pr_auc   roc_auc                                                                                                                                            secondary_metrics  adaptation_count  latency_p50_ms  latency_p95_ms  latency_p99_ms  throughput_tx_per_sec                                                                     artifact_path error_message  window_size  no_swap
0  IEEE-CIS_Objective1_E1_E2_P1_W10000_seed42  Objective1_E1_E2  IEEE-CIS     P1    42  COMPLETED  2026-09-13T12:51:20.469466+00:00  2026-09-13T12:51:27.978526+00:00                  7.301    af2d52cfe83db380       authoritative_o1_v3_m1_m2_m3  0.012738  0.323713  {'n_samples': 850, 'n_positive': 15, 'pr_auc': 0.012738272782573319, 'roc_auc': 0